# 🏁 Week 8 — Final Projects
> **Goal:** Build a complete, working AI application using everything learned in Weeks 1-7.

---
This notebook contains **4 full project implementations**. Pick one (or build all 4):

| Project | Description | Tech |
|---|---|---|
| **A** | AI SaaS Backend | FastAPI + PydanticAI + Groq |
| **B** | Document Analysis System | Multi-agent pipeline |
| **C** | AI Customer Support Bot | Tool-calling + routing |
| **D** | Multi-Model Research Assistant | OpenRouter + all models |

## Setup

In [ ]:
%pip install pydantic-ai groq openai fastapi uvicorn python-dotenv httpx rich --quiet

In [ ]:
import os, sys
sys.path.insert(0, ".")
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY!"
print("Ready.")

---
# 🅰 Project A — AI SaaS Backend
> FastAPI server that exposes AI endpoints. Clients send requests, get typed JSON back.

In [ ]:
# ── schemas.py (inline for notebook) ─────────────────────────────────────
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

class AnalyseRequest(BaseModel):
    text: str = Field(..., min_length=10, max_length=10_000)
    analysis_type: str = Field(default="general", description="general | sentiment | summary | keywords")

class AnalyseResponse(BaseModel):
    analysis_type: str
    result: str
    keywords: List[str]
    confidence: float = Field(ge=0.0, le=1.0)
    word_count: int

class ChatRequest(BaseModel):
    message: str
    session_id: Optional[str] = None

class ChatResponse(BaseModel):
    reply: str
    session_id: str
    model_used: str


# ── FastAPI app ────────────────────────────────────────────────────────────
from fastapi import FastAPI, HTTPException
from pydantic_ai import Agent
import uuid
import sys
sys.path.insert(0, "..")
from week7_production.cache import ResponseCache

app = FastAPI(title="AI SaaS API", version="1.0.0")
cache = ResponseCache(ttl_seconds=600)
sessions: dict = {}  # in production: Redis

analyse_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    result_type=AnalyseResponse,
    system_prompt=(
        "You are a text analysis API. Analyse the given text as requested. "
        "Return structured analysis with keywords and confidence."
    ),
)

chat_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="You are a helpful AI assistant. Be concise and friendly.",
)

@app.post("/analyse", response_model=AnalyseResponse)
async def analyse_text(request: AnalyseRequest) -> AnalyseResponse:
    key = cache.make_key("analyse", request.text + request.analysis_type)
    cached = cache.get(key)
    if cached:
        return AnalyseResponse(**cached) if isinstance(cached, dict) else cached

    try:
        result = await analyse_agent.run(
            f"Analysis type: {request.analysis_type}\nText: {request.text}"
        )
        cache.set(key, result.data.model_dump())
        return result.data
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest) -> ChatResponse:
    session_id = request.session_id or str(uuid.uuid4())
    history = sessions.get(session_id, [])

    result = await chat_agent.run(request.message, message_history=history)
    sessions[session_id] = result.new_messages()

    return ChatResponse(
        reply=result.data,
        session_id=session_id,
        model_used="groq:llama-3.1-8b-instant",
    )

@app.get("/health")
async def health():
    return {"status": "ok", "cache": cache.stats()}


print("FastAPI app defined. To run:")
print("  uvicorn week8_final_projects.08_final_projects:app --reload")
print("\nEndpoints:")
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f"  {list(route.methods)[0]:<6} {route.path}")

In [ ]:
# Test Project A locally (without running the server)
from fastapi.testclient import TestClient

# Note: TestClient is sync — for async endpoints use httpx.AsyncClient
# This demo tests the schema/structure
req = AnalyseRequest(text="Python is a popular programming language used in AI and data science.", analysis_type="keywords")
print("Request schema valid:", req.model_dump())

# Direct agent call to demo the output
result = await analyse_agent.run(f"Analysis type: {req.analysis_type}\nText: {req.text}")
print(f"\nAnalysis result:")
print(f"  Keywords  : {result.data.keywords}")
print(f"  Confidence: {result.data.confidence}")
print(f"  Result    : {result.data.result}")

---
# 🅱 Project B — Document Analysis System

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from pydantic_ai import Agent
from project_helpers import MOCK_DOCUMENTS, save_json, timestamp
import asyncio

# ── Schemas ────────────────────────────────────────────────────────────────
class DocumentInsight(BaseModel):
    doc_id: str
    title: str
    doc_type: str
    key_points: List[str] = Field(min_length=2)
    sentiment: str = Field(description="positive | negative | neutral | mixed")
    action_items: List[str]
    urgency: str = Field(description="low | medium | high | critical")
    summary: str

class PortfolioReport(BaseModel):
    analysis_date: str
    total_documents: int
    critical_items: List[str]
    overall_health: str = Field(description="green | yellow | red")
    executive_summary: str
    recommended_actions: List[str]


# ── Agents ─────────────────────────────────────────────────────────────────
doc_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    result_type=DocumentInsight,
    system_prompt=(
        "You are a business analyst. Analyse the document and extract "
        "structured insights, action items, and urgency level."
    ),
)

portfolio_agent = Agent(
    model="groq:llama-3.1-70b-versatile",
    result_type=PortfolioReport,
    system_prompt=(
        "You are a senior business analyst. Given multiple document analyses, "
        "synthesise them into an executive portfolio report."
    ),
)


# ── Pipeline ───────────────────────────────────────────────────────────────
async def analyse_document(doc: dict) -> DocumentInsight:
    prompt = f"Document ID: {doc['id']}\nTitle: {doc['title']}\nType: {doc['type']}\nContent: {doc['content']}"
    result = await doc_agent.run(prompt)
    return result.data


import json
# Analyse all documents in parallel
insights = await asyncio.gather(*[analyse_document(d) for d in MOCK_DOCUMENTS])

print("Individual Document Insights:")
for insight in insights:
    urgency_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(insight.urgency, "⚪")
    print(f"  {urgency_icon} [{insight.urgency.upper()}] {insight.title}")
    print(f"     Sentiment : {insight.sentiment}")
    print(f"     Actions   : {len(insight.action_items)} items")
    print(f"     Summary   : {insight.summary[:80]}...")
    print()

# Synthesise into portfolio report
insights_json = json.dumps([i.model_dump() for i in insights], indent=2)
portfolio_result = await portfolio_agent.run(
    f"Analyse these {len(insights)} document insights and create an executive report:\n{insights_json}"
)
portfolio: PortfolioReport = portfolio_result.data

health_icon = {"green": "✅", "yellow": "⚠️", "red": "🚨"}.get(portfolio.overall_health, "❓")
print(f"\n{'='*60}")
print(f"PORTFOLIO REPORT {health_icon} {portfolio.overall_health.upper()}")
print(f"{'='*60}")
print(f"Executive Summary: {portfolio.executive_summary}")
print(f"\nRecommended Actions:")
for a in portfolio.recommended_actions:
    print(f"  → {a}")

# Save report
saved = save_json(portfolio, f"portfolio_report_{timestamp()}.json")
print(f"\nReport saved: {saved}")

---
# 🅲 Project C — AI Customer Support Bot

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from pydantic_ai import Agent
from project_helpers import MOCK_TICKETS
import asyncio

# ── Schemas ────────────────────────────────────────────────────────────────
class TicketClassification(BaseModel):
    ticket_id: str
    category: str = Field(description="billing | auth | api | feature_request | outage | other")
    priority: str = Field(description="low | medium | high | critical")
    sentiment: str = Field(description="frustrated | neutral | polite")
    requires_human: bool
    auto_reply: Optional[str] = Field(description="Auto-reply if this can be resolved automatically")
    escalation_reason: Optional[str] = Field(description="Why it needs human agent, if applicable")

class SupportResponse(BaseModel):
    ticket_id: str
    response: str
    suggested_kb_articles: List[str]
    next_steps: List[str]


# ── Agents ─────────────────────────────────────────────────────────────────
classifier_agent = Agent(
    model="groq:llama-3.1-8b-instant",     # cheap for classification
    result_type=TicketClassification,
    system_prompt=(
        "You are a customer support triage system. "
        "Classify incoming tickets by category, priority, and sentiment. "
        "Auto-resolve simple issues where possible."
    ),
)

response_agent = Agent(
    model="groq:llama-3.1-70b-versatile",  # better for writing responses
    result_type=SupportResponse,
    system_prompt=(
        "You are a helpful customer support representative. "
        "Write professional, empathetic responses. "
        "Suggest relevant knowledge base articles."
    ),
)


# ── Pipeline ───────────────────────────────────────────────────────────────
async def process_ticket(ticket: dict) -> tuple[TicketClassification, Optional[SupportResponse]]:
    # Step 1: Classify
    classify_prompt = f"Ticket ID: {ticket['id']}\nUser: {ticket['user']}\nMessage: {ticket['message']}"
    classify_result = await classifier_agent.run(classify_prompt)
    classification = classify_result.data
    classification.ticket_id = ticket["id"]  # ensure ID is set

    # Step 2: Generate response (always generate, human will review if needed)
    response_prompt = f"""
    Ticket ID  : {ticket['id']}
    User message: {ticket['message']}
    Category   : {classification.category}
    Priority   : {classification.priority}
    
    Write a helpful response.
    """
    response_result = await response_agent.run(response_prompt)
    return classification, response_result.data


# Process all tickets
results = await asyncio.gather(*[process_ticket(t) for t in MOCK_TICKETS])

print("SUPPORT QUEUE RESULTS")
print("=" * 70)
for i, (cls, resp) in enumerate(results):
    ticket = MOCK_TICKETS[i]
    priority_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(cls.priority, "⚪")
    human_icon = "👤" if cls.requires_human else "🤖"

    print(f"\n{priority_icon} [{cls.priority.upper()}] {ticket['id']} | {cls.category} | {human_icon}")
    print(f"User: {ticket['message'][:80]}...")
    if cls.auto_reply:
        print(f"Auto-reply: {cls.auto_reply[:100]}")
    elif resp:
        print(f"Draft response: {resp.response[:100]}...")
    if cls.requires_human:
        print(f"Escalate because: {cls.escalation_reason}")

# Stats
auto_resolved = sum(1 for cls, _ in results if not cls.requires_human)
critical = sum(1 for cls, _ in results if cls.priority == "critical")
print(f"\n{'='*70}")
print(f"Total tickets  : {len(results)}")
print(f"Auto-resolved  : {auto_resolved} ({auto_resolved/len(results)*100:.0f}%)")
print(f"Needs human    : {len(results) - auto_resolved}")
print(f"Critical issues: {critical}")

---
# 🅳 Project D — Multi-Model Research Assistant

In [ ]:
import os
import asyncio
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from openai import AsyncOpenAI
from project_helpers import save_json, timestamp

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# ── Schemas ────────────────────────────────────────────────────────────────
class ResearchAngle(BaseModel):
    model_used: str
    perspective: str = Field(description="technical | business | academic | practical")
    key_insights: List[str] = Field(min_length=3)
    confidence: int = Field(ge=1, le=10)

class MultiModelReport(BaseModel):
    topic: str
    angles: List[ResearchAngle]
    consensus_points: List[str]
    disagreements: List[str]
    final_recommendation: str
    best_sources_to_read: List[str]


# ── Multi-model research ────────────────────────────────────────────────────
async def research_with_model(
    topic: str,
    perspective: str,
    model_id: str,
    use_openrouter: bool = False,
) -> ResearchAngle:
    """Research a topic from a specific perspective using a specific model."""

    system_prompt = (
        f"You are a {perspective} expert researcher. "
        f"Analyse the given topic from a {perspective} perspective. "
        f"Return your top insights and confidence level."
    )

    if use_openrouter and OPENROUTER_API_KEY:
        client = AsyncOpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1",
        )
        model = OpenAIModel(model_id, openai_client=client)
    else:
        model = f"groq:{model_id}"  # fallback to Groq

    agent = Agent(
        model=model,
        result_type=ResearchAngle,
        system_prompt=system_prompt,
    )

    result = await agent.run(f"Research topic: {topic}")
    result.data.model_used = model_id
    result.data.perspective = perspective
    return result.data


# ── Synthesizer agent ──────────────────────────────────────────────────────
synthesizer = Agent(
    model="groq:llama-3.1-70b-versatile",
    result_type=MultiModelReport,
    system_prompt=(
        "You are a meta-analyst. Given research from multiple models and perspectives, "
        "synthesise a balanced final report identifying consensus and disagreements."
    ),
)


# ── Run the multi-model research ────────────────────────────────────────────
topic = "Should startups build AI features using API-based LLMs or fine-tuned local models?"

# Research same topic from 4 perspectives using different models
research_tasks = [
    research_with_model(topic, "technical",  "llama-3.1-70b-versatile"),
    research_with_model(topic, "business",   "llama-3.1-8b-instant"),
    research_with_model(topic, "practical",  "llama-3.1-8b-instant"),
    research_with_model(topic, "academic",   "llama-3.1-70b-versatile"),
]

angles = await asyncio.gather(*research_tasks)

print(f"Got {len(angles)} research angles")
for a in angles:
    print(f"  [{a.perspective:<10}] confidence={a.confidence}/10 | model={a.model_used}")

# Synthesise all angles
angles_json = json.dumps([a.model_dump() for a in angles], indent=2)
synth_result = await synthesizer.run(
    f"Topic: {topic}\n\nResearch angles:\n{angles_json}\n\nSynthesise into a final report."
)
report: MultiModelReport = synth_result.data

from rich.console import Console
from rich.panel import Panel

console = Console()
console.print(Panel(f"[bold cyan]Multi-Model Research Report[/]"))
console.print(f"[bold]Topic:[/] {report.topic}")
console.print(f"\n[bold]Consensus Points:[/]")
for p in report.consensus_points: console.print(f"  ✅ {p}")
console.print(f"\n[bold]Disagreements:[/]")
for d in report.disagreements: console.print(f"  ⚖️  {d}")
console.print(f"\n[bold]Final Recommendation:[/]\n{report.final_recommendation}")
console.print(f"\n[bold]Best Sources to Read:[/]")
for s in report.best_sources_to_read: console.print(f"  📚 {s}")

saved = save_json(report, f"multimodel_report_{timestamp()}.json")
console.print(f"\n[green]Saved: {saved}[/]")

---
## 🎓 Course Complete!

### What you mastered across 8 weeks:

| Week | Topic | Key Skill |
|------|-------|-----------|
| 1 | Pydantic Basics | Type-safe data validation |
| 2 | PydanticAI Core | Agents with structured outputs |
| 3 | Groq Integration | Ultra-fast streaming inference |
| 4 | OpenRouter | Multi-model routing & cost control |
| 5 | Tool Calling | Agents with real-world tools |
| 6 | Multi-Agent Systems | Pipelines with specialised agents |
| 7 | Production Design | Caching, logging, rate limiting |
| 8 | Final Projects | End-to-end AI applications |

### Next steps:
- Deploy Project A (FastAPI) to Railway / Fly.io / AWS
- Add a real vector database (Qdrant, Pinecone) to the research agent
- Swap the in-memory cache for Redis
- Ship it! 🚀